In [ ]:
!pwd
!hostname
!which python

In [ ]:
import pandas as pd
from datetime import datetime
import random
random.seed(42)

In [ ]:
from vllm import LLM, SamplingParams
import torch
import gc

print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.device_count())

In [ ]:
llm_path = "/gpfs/projects/bsc100/models/meta-llama/Llama-3.1-8B-Instruct"
timestamp = datetime.now().strftime("%Y%m%d_%H%M")
llm = LLM(model=llm_path)

In [ ]:
metadata_df = pd.read_csv('/gpfs/projects/bsc100/textmachine-data/preprocessed_data/output_blmicrosoft/deduplicated_metadata/metadata_deduplicated_tm57.csv').iloc[1000:1050]

In [ ]:
metadata_df.iloc[0]

In [ ]:
fiction_titles = [
    "The white doe of Rylstone; or The fate of the Nortons. A poem by Wordsworth, William",
    "Oliver Twist by Dickens, Charles",
    "Emma by Austen, Jane",
    "The Luck of Barry Lyndon, a romance of the last century by Thackeray, William Makepeace",
    "The Bride of Abydos. A Turkish tale by Byron, George Gordon Byron, Baron",
]

nonfiction_titles = [
    "Ceylon in 1893, etc by Ferguson, John",
    "Memoirs of the principal events in the campaigns of North Holland and Egypt: together with a brief description of the Islands of Crete, Rhodes, Syracuse, etc by Maule, Francis",
    "The Campaigns in Virginia 1861-62. Reprinted from the 'Illustrated Naval and Military Magazine.' by Maguire, T. Miller (Thomas Miller)",
    "The Tourists' Handy Guide to Scotland ... Twelfth edition ... enlarged by Scotland",
    "Among the Goths and Vandals [On Sweden.] by Blaikie, John",
]

In [ ]:
def generate_user_prompt_zeroshot(
    fiction_titles, nonfiction_titles, target_title, target_author
):
    s = f"""Given a title and its author, return only the 'fiction' or 'non-fiction' label, without any additional explanation.
    
    Examples of genres that should be labeled as fiction are: poetry, novels, romances, plays, comedies, tragedies, recitations, short stories, tales, songs, operas, odes, poems.
    Examples of genres that should be labeled as non-fiction are: memoirs, biographies, autobiographies, essays, history books, travelogues, textbooks, guidebooks, scientific studies, philosophical treatises, and commentaries.

    Classify the following title as either "fiction" or "non-fiction": \n{target_title} by {target_author}.

    Label: 
    """
    return s


def generate_user_prompt_fewshot(
    fiction_titles, nonfiction_titles, target_title, target_author
):
    s = f"""Given a title and its author, return only the 'fiction' or 'non-fiction' label, without any additional explanation.
    
    Examples of genres that should be labeled as fiction are: poetry, novels, romances, plays, comedies, tragedies, recitations, short stories, tales, songs, operas, odes, poems.
    Examples of genres that should be labeled as non-fiction are: memoirs, biographies, autobiographies, essays, history books, travelogues, textbooks, guidebooks, scientific studies, philosophical treatises, and commentaries.

    Below are five examples of fiction book titles:\n{"\n".join(fiction_titles)}
    
    Below are five examples of non-fiction book titles:\n{"\n".join(nonfiction_titles)}

    Classify the following title as either "fiction" or "non-fiction": \n{target_title} by {target_author}.

    Label: 
    """
    return s

In [ ]:
system_prompt = """
    You are a helpful assistant that classifies books as either fiction or non-fiction based on their title. 
    """

# create the user prompt template, where we will later fill in the fiction and non-fiction shots and the example title to classify
metadata_df["user_prompt"] = metadata_df.apply(
    lambda x: generate_user_prompt_zeroshot(
        fiction_titles, nonfiction_titles, x["title"], x["author"]
    ),
    axis=1,
)

In [ ]:
print(metadata_df.iloc[0]["user_prompt"])

In [ ]:
def classify_genre(
    df,
    system_prompt,
    user_prompt_col,
    output_col,
    batch_size=10,
    sampling_params=dict(),
):
    df = df.copy()

    sampling = SamplingParams(
        temperature=sampling_params["temperature"],
        top_p=sampling_params["top_p"],
        repetition_penalty=sampling_params["repetition_penalty"],
        max_tokens=128,
        seed=42,
    )

    df[output_col] = None
    total = len(df)
    
    for start in range(0, total, batch_size):
        end = min(start + batch_size, total)
        batch = df.iloc[start:end]

        messages = [
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": str(text)},
            ]
            for text in batch[user_prompt_col]
        ]

        print(messages)

        outputs = llm.chat(messages=messages, sampling_params=sampling)

        for i, output in enumerate(outputs):
            output_text = output.outputs[0].text.strip()
            df.at[batch.index[i], output_col] = output_text

        print(f"Processed rows {start+1}-{end}/{total}")

    return df[output_col]

In [ ]:
configs = {"temperature": 0, "top_p": 0.9, "repetition_penalty": 1}

output_col = "genre_llm"
metadata_df[output_col] = classify_genre(
    metadata_df,
    system_prompt=system_prompt,
    user_prompt_col="user_prompt",
    output_col=output_col,
    batch_size=20,
    sampling_params=configs,
)

# pd.DataFrame(data=metadata_df).to_csv(
#     os.path.join("metadata_df_llm.csv"),
#     index=False,
# )

In [ ]:
metadata_df[["title", "author", "genre_llm"]]

In [ ]:
del llm
gc.collect()
torch.cuda.empty_cache()
print("GPU memory released")